<div align="center">

# 🍋 SMARTFRUIT
## Prototipo V0 · Vision AI
### DINOv2-small + Logistic Regression

**Clasificación de oleocelosis en imágenes individuales de limón**

</div>

---

> **Objetivo del notebook**  
> Organizar y reproducir el experimento V0: auditoría de etiquetas, extracción de embeddings con DINOv2, entrenamiento del clasificador, evaluación, predicción interactiva y exportación del contrato JSON.

| Componente | Implementación |
|---|---|
| Extractor visual | `facebook/dinov2-small` congelado |
| Clasificador | `StandardScaler` + `LogisticRegression` |
| Unidad analizada | Una imagen/crop individual |
| Salida | Clase estimada, scores y JSON de integración |

**Alcance y cautelas**
- La etiqueta se deriva de las anotaciones YOLO: si aparece clase `1`, se considera “con oleocelosis”; si no, y aparece clase `0`, se considera “sin oleocelosis”.
- Las métricas reflejan concordancia con esas etiquetas derivadas; **no constituyen diagnóstico clínico ni probabilidades calibradas**.
- Este V0 clasifica crops individuales. No detecta automáticamente todos los limones en una foto grupal.
- **SAM 3 aún no está integrado** en este notebook.
- Se conserva la separación `train` / `val` / `test`: el test es la evaluación final, no una fuente para ajustar el modelo.



## 🧭 Ruta de trabajo

1. Preparar entorno y localizar proyecto/dataset.
2. Construir etiquetas derivadas de anotaciones YOLO y revisar los splits.
3. Extraer embeddings DINOv2 y reutilizar caché cuando exista.
4. Ajustar Logistic Regression y evaluar en validación y test.
5. Guardar artefactos, probar imágenes con un selector interactivo y exportar JSON.

> **Entorno previsto:** Google Colab con Google Drive montado. La primera ejecución puede descargar el modelo DINOv2 y tardar más; ejecuciones posteriores pueden reutilizar los embeddings guardados.


## 1 · ⚙️ Entorno y rutas

In [ ]:
from pathlib import Path
import json, os, random
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive no disponible o no es Colab:", repr(e))

MYDRIVE = Path("/content/drive/MyDrive")
expected = MYDRIVE / "MAESTRÍA UTEC" / "CICLO V" / "PROYECTO INTEGRADOR" / "SMARTFRUIT BASELINE"
if expected.exists():
    PROJECT_DIR = expected
else:
    matches = list(MYDRIVE.glob("**/SMARTFRUIT BASELINE")) if MYDRIVE.exists() else []
    if not matches:
        raise FileNotFoundError("No se encontró SMARTFRUIT BASELINE en MyDrive.")
    PROJECT_DIR = matches[0]
DATASET_DIR = PROJECT_DIR / "dataset_oleocelosis_yolo_v3_reproducible"
if not DATASET_DIR.exists():
    found = list(PROJECT_DIR.glob("**/dataset_oleocelosis_yolo_v3_reproducible"))
    if not found:
        raise FileNotFoundError("No se encontró dataset_oleocelosis_yolo_v3_reproducible.")
    DATASET_DIR = found[0]
print("Proyecto:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)

## 2 · 🏷️ Etiquetas derivadas de anotaciones YOLO

In [ ]:
from PIL import Image
IMG_EXTS = {".jpg",".jpeg",".png",".webp"}
def label_from_yolo(path):
    if not path.exists(): return None, "sin_archivo_label"
    classes = []
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if not line.strip(): continue
        try: classes.append(int(float(line.split()[0])))
        except Exception: return None, "linea_malformada"
    if not classes: return None, "label_vacio"
    if 1 in classes: return 1, "contiene_clase_1"
    if 0 in classes: return 0, "solo_clase_0"
    return None, "clase_desconocida"

records, excluded = [], []
for split in ["train","val","test"]:
    imdir, lbdir = DATASET_DIR/"images"/split, DATASET_DIR/"labels"/split
    for p in sorted(imdir.glob("*")) if imdir.exists() else []:
        if p.suffix.lower() not in IMG_EXTS: continue
        y, reason = label_from_yolo(lbdir/(p.stem+".txt"))
        if y is None: excluded.append({"split":split,"image":p.name,"reason":reason})
        else: records.append({"split":split,"image_path":str(p),"image_name":p.name,"y":y,"reason":reason})
df = pd.DataFrame(records)
print("Imágenes utilizables:",len(df),"| excluidas:",len(excluded))
display(df.groupby(["split","y"]).size().unstack(fill_value=0).rename(columns={0:"sin",1:"con"}))
if excluded: display(pd.DataFrame(excluded).head(10))

## 3 · 🔎 Auditoría de los splits

In [ ]:
for split in ["train","val","test"]:
    part = df[df.split == split]
    print(split, "n =",len(part),"| sin =",int((part.y==0).sum()),"| con =",int((part.y==1).sum()))
name_splits = df.groupby("image_name")["split"].nunique()
print("Nombres repetidos entre splits:",int((name_splits>1).sum()))

## 4 · 📦 Dependencias

In [ ]:
%pip -q install transformers scikit-learn joblib tqdm

## 5 · 🧠 Cargar DINOv2-small y definir extracción de embeddings

In [ ]:
import torch, joblib
from transformers import AutoImageProcessor, AutoModel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/dinov2-small"
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
dino = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
dino.eval()
print("Dispositivo:",DEVICE)
def extract_embeddings(frame, batch_size=16):
    vectors, labels, paths = [], [], []
    for start in tqdm(range(0,len(frame),batch_size)):
        chunk = frame.iloc[start:start+batch_size]
        images = [Image.open(p).convert("RGB") for p in chunk.image_path]
        inputs = processor(images=images,return_tensors="pt")
        inputs = {k:v.to(DEVICE) for k,v in inputs.items()}
        with torch.no_grad():
            output = dino(**inputs)
            emb = output.last_hidden_state[:,0,:].cpu().numpy()
        vectors.append(emb); labels.extend(chunk.y.astype(int).tolist()); paths.extend(chunk.image_path.tolist())
    if not vectors: return np.empty((0,384),dtype=np.float32),np.array([],dtype=int),[]
    return np.concatenate(vectors).astype(np.float32),np.asarray(labels,dtype=int),paths

## 6 · 💾 Extraer o reutilizar características por split

In [ ]:
OUTPUT_DIR = PROJECT_DIR/"PROTOTIPO_V0_DINOV2"/"resultados"
FEATURE_DIR = OUTPUT_DIR/"caracteristicas_dinov2"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); FEATURE_DIR.mkdir(parents=True,exist_ok=True)
REUSE_FEATURES = True

def get_features(split):
    part = df[df.split==split].reset_index(drop=True)
    cache = FEATURE_DIR/f"{split}_dinov2.npz"
    if REUSE_FEATURES and cache.exists():
        z=np.load(cache,allow_pickle=True)
        print("Reutilizando",cache.name)
        return z["X"],z["y"],z["paths"].tolist()
    X,y,paths=extract_embeddings(part)
    np.savez_compressed(cache,X=X,y=y,paths=np.asarray(paths,dtype=str))
    return X,y,paths

X_train,y_train,paths_train=get_features("train")
X_val,y_val,paths_val=get_features("val")
X_test,y_test,paths_test=get_features("test")
print("Shapes:",X_train.shape,X_val.shape,X_test.shape)

## 7 · 🧪 Entrenamiento y validación

In [ ]:
if not len(y_train) or not len(y_val) or not len(y_test):
    raise ValueError("Algún split está vacío; revisa etiquetas y dataset.")
classifier=Pipeline([
    ("scaler",StandardScaler()),
    ("logreg",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED))
])
classifier.fit(X_train,y_train)

def evaluate(model,X,y,name):
    pred=model.predict(X); probs=model.predict_proba(X)[:,1]
    metrics={
      "split":name,"n":int(len(y)),
      "accuracy":float(accuracy_score(y,pred)),
      "precision":float(precision_score(y,pred,zero_division=0)),
      "recall":float(recall_score(y,pred,zero_division=0)),
      "f1":float(f1_score(y,pred,zero_division=0)),
      "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
      "roc_auc":float(roc_auc_score(y,probs)) if len(np.unique(y))==2 else None,
      "confusion_matrix":confusion_matrix(y,pred,labels=[0,1]).tolist()
    }
    print("\\n",name.upper())
    print(classification_report(y,pred,labels=[0,1],target_names=["sin_oleocelosis","con_oleocelosis"],zero_division=0))
    return metrics
val_metrics=evaluate(classifier,X_val,y_val,"val")
display(pd.DataFrame([{k:v for k,v in val_metrics.items() if k!="confusion_matrix"}]))

## 8 · 📊 Evaluación final en test

In [ ]:
test_metrics=evaluate(classifier,X_test,y_test,"test")
display(pd.DataFrame(test_metrics["confusion_matrix"],index=["real_sin","real_con"],columns=["pred_sin","pred_con"]))
display(pd.DataFrame([{k:v for k,v in test_metrics.items() if k!="confusion_matrix"}]))

## 9 · 💿 Guardar clasificador y métricas

In [ ]:
MODEL_PATH=OUTPUT_DIR/"dinov2_logreg_v0.joblib"
METRICS_PATH=OUTPUT_DIR/"metricas_dinov2_v0.json"
joblib.dump(classifier,MODEL_PATH)
payload={
 "model":MODEL_NAME,"classifier":"StandardScaler + LogisticRegression","seed":SEED,
 "label_rule":"class 1 present => con oleocelosis; otherwise class 0 present => sin oleocelosis",
 "validation_metrics":val_metrics,"test_metrics":test_metrics,
 "notes":["Evaluación contra etiquetas derivadas de anotaciones YOLO.",
          "No equivale a diagnóstico clínico ni a probabilidad calibrada.",
          "Clasifica imágenes individuales; no detecta frutos en fotos grupales.",
          "SAM 3 aún no está integrado."]
}
METRICS_PATH.write_text(json.dumps(payload,indent=2,ensure_ascii=False),encoding="utf-8")
print("Modelo:",MODEL_PATH)
print("Métricas:",METRICS_PATH)

## 10 · 🖼️ Predicción individual

In [ ]:
def predecir_oleocelosis(image_path):
    image=Image.open(image_path).convert("RGB")
    inputs=processor(images=image,return_tensors="pt")
    inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.no_grad():
        output=dino(**inputs)
        emb=output.last_hidden_state[:,0,:].cpu().numpy()
    probs=classifier.predict_proba(emb)[0]
    pred=int(np.argmax(probs))
    return {
      "image_id":Path(image_path).name,
      "predicted_class":pred,
      "label":"con_oleocelosis" if pred==1 else "sin_oleocelosis",
      "p_con_oleocelosis":float(probs[1]),
      "p_sin_oleocelosis":float(probs[0]),
      "confidence":float(np.max(probs)),
      "model_version":"dinov2_logreg_v0"
    }

IMAGE_TO_TEST=""
if IMAGE_TO_TEST:
    print(json.dumps(predecir_oleocelosis(IMAGE_TO_TEST),indent=2,ensure_ascii=False))
else:
    print("Completa IMAGE_TO_TEST con la ruta de un crop para probar.")

## 11 · 🔗 Contrato JSON para integración de Vision AI

In [ ]:
def analizar_muestra_json(image_paths,sample_id="MUESTRA_DEMO_001"):
    results,errors=[],[]
    for i,path in enumerate(image_paths,1):
        try:
            r=predecir_oleocelosis(path); r["crop_id"]=f"CROP_{i:03d}"; results.append(r)
        except Exception as e:
            errors.append({"image_id":str(path),"error":str(e)})
    total=len(results); affected=sum(r["predicted_class"]==1 for r in results)
    return {
      "schema_version":"1.0","component":"vision_ai","image_id":sample_id,
      "vision_status":"success" if total and not errors else ("partial" if total else "error"),
      "total_fruits":total,"oleocellosis_count":int(affected),
      "oleocellosis_pct":round(100*affected/total,2) if total else None,
      "mean_confidence":float(np.mean([r["confidence"] for r in results])) if total else None,
      "model_version":"dinov2_logreg_v0","fruit_results":results,"errors":errors
    }

DEMO_IMAGE_PATHS = uploaded_image_paths if "uploaded_image_paths" in globals() else []  # O asigna rutas manualmente.
if DEMO_IMAGE_PATHS:
    demo=analizar_muestra_json(DEMO_IMAGE_PATHS)
    dest=PROJECT_DIR/"servicio_vision"/"vision_output_demo.json"
    dest.parent.mkdir(parents=True,exist_ok=True)
    dest.write_text(json.dumps(demo,indent=2,ensure_ascii=False),encoding="utf-8")
    print("JSON guardado en",dest)
    print(json.dumps(demo,indent=2,ensure_ascii=False))
else:
    print("Agrega crops reales a DEMO_IMAGE_PATHS para exportar el JSON.")

## 12 · 🚀 Demo interactiva: seleccionar una imagen

Esta celda agrega el botón de carga de archivos de Google Colab. Selecciona uno o varios crops de limón (`.jpg`, `.jpeg`, `.png` o `.webp`); el modelo mostrará la imagen, la clase estimada, los puntajes y el JSON resultante.

**Importante:** la primera vez debes ejecutar previamente las secciones 1–10 para cargar el dataset, extraer características y entrenar el clasificador.


In [ ]:
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

uploaded_image_paths = []

try:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()  # Abre el selector de archivos de Colab
    for filename, content in uploaded.items():
        suffix = Path(filename).suffix.lower()
        if suffix not in IMG_EXTS:
            print(f"Se omite {filename}: formato no compatible.")
            continue
        # Guardar temporalmente la imagen cargada para usar el mismo pipeline
        target = Path("/content") / Path(filename).name
        target.write_bytes(content)
        uploaded_image_paths.append(str(target))

    if not uploaded_image_paths:
        print("No se cargaron imágenes compatibles.")
    else:
        print(f"Imágenes cargadas: {len(uploaded_image_paths)}")
        demo_json = analizar_muestra_json(uploaded_image_paths, sample_id="SMARTFRUIT_DEMO")
        for path, result in zip(uploaded_image_paths, demo_json["fruit_results"]):
            display(Markdown(f"### {Path(path).name}"))
            image = Image.open(path).convert("RGB")
            plt.figure(figsize=(6, 4))
            plt.imshow(image)
            plt.axis("off")
            plt.title(result["label"].replace("_", " ").title())
            plt.show()
            display(pd.DataFrame([{
                "Predicción": result["label"],
                "P(sin oleocelosis)": round(result["p_sin_oleocelosis"], 4),
                "P(con oleocelosis)": round(result["p_con_oleocelulosis"], 4),
                "Confianza máxima": round(result["confidence"], 4)
            }]))

        display(Markdown("### Resumen JSON"))
        print(json.dumps(demo_json, indent=2, ensure_ascii=False))

        json_dest = PROJECT_DIR / "servicio_vision" / "vision_output_demo.json"
        json_dest.parent.mkdir(parents=True, exist_ok=True)
        json_dest.write_text(json.dumps(demo_json, indent=2, ensure_ascii=False), encoding="utf-8")
        print("JSON guardado en:", json_dest)
        display(Markdown("Puedes descargar el JSON desde el panel de archivos de Colab o ejecutar la celda de exportación al final."))
except ImportError:
    print("El selector interactivo requiere Google Colab. En Jupyter local, asigna la ruta de la imagen a IMAGE_TO_TEST en la sección 10.")


## 13 · ✅ Conclusiones y alcance

- **Baseline:** YOLO11n detector; reporta Precision, Recall y mAP.
- **V0:** DINOv2-small congelado + regresión logística; clasifica imágenes individuales y reporta Accuracy, F1 y ROC-AUC.
- No son métricas directamente comparables: son tareas distintas.
- Train se usa para ajustar, validation para revisar y test para evaluación final.
- `oleocellosis_pct` es el porcentaje de crops clasificados como afectados en la muestra procesada; no es prevalencia real si el muestreo no es representativo.
- **Pendiente:** detección/segmentación de frutos en imágenes grupales (SAM 3 u otro detector) e integración end-to-end.

### Artefactos generados

- `resultados/caracteristicas_dinov2/{train,val,test}_dinov2.npz`
- `resultados/dinov2_logreg_v0.joblib`
- `resultados/metricas_dinov2_v0.json`
- `servicio_vision/vision_output_demo.json`

### Lectura responsable de los resultados

Los scores `p_con_oleocelulosis` y `p_sin_oleocelulosis` son salidas del clasificador, no probabilidades clínicas calibradas. Una predicción sobre una sola imagen o una pequeña muestra es una demostración funcional, no una estimación validada de prevalencia del lote.

**Siguiente paso técnico:** integrar un módulo que detecte/segmente cada fruta en la foto grupal, generar crops trazables a la imagen original y validar el flujo completo con fotos nuevas y etiquetas revisadas.
